# Catalogs-Schemas-Tables

In [0]:
%sql
SELECT table_schema, table_name, table_type
FROM nutrichain_lakehouse.information_schema.tables
WHERE table_schema IN ('bronze', 'silver', 'gold')
ORDER BY table_schema, table_name;

In [0]:
%sql
SELECT primary_country, COUNT(DISTINCT country_iso_code) AS iso_codes
FROM nutrichain_lakehouse.silver.silver_openfood_products
GROUP BY primary_country
HAVING COUNT(DISTINCT country_iso_code) > 1
ORDER BY iso_codes DESC
LIMIT 20;

In [0]:
%sql
SELECT table_catalog, table_schema, table_name, table_type
FROM nutrichain_lakehouse.information_schema.tables
WHERE table_schema IN ('bronze', 'silver', 'gold')
ORDER BY table_schema, table_name;

# Pipeline Health

In [0]:
%sql
SELECT *
FROM nutrichain_lakehouse.gold.pipeline_audit
ORDER BY run_ingested_at DESC
LIMIT 10;

# Bronze: page rows per batch

In [0]:
%sql
SELECT
    ingest_run_id,
    COUNT(*) AS page_rows,
    SUM(page_size) AS products_in_pages,
    MIN(ingested_at) AS first_ingested_at,
    MAX(ingested_at) AS last_ingested_at
FROM nutrichain_lakehouse.bronze.bronze_openfood_products_raw
GROUP BY ingest_run_id
ORDER BY last_ingested_at DESC;

# Bronze: Latest batch only

In [0]:
%sql
SELECT
    ingest_run_id,
    COUNT(*) AS page_rows
FROM nutrichain_lakehouse.bronze.bronze_openfood_products_raw
WHERE ingest_run_id = (
    SELECT ingest_run_id
    FROM nutrichain_lakehouse.bronze.bronze_openfood_products_raw
    ORDER BY ingested_at DESC
    LIMIT 1
)
GROUP BY ingest_run_id;


# Bronze: Ingestion Page rows Check

In [0]:
%sql
WITH latest_run AS (
  SELECT MAX(ingest_run_id) AS ingest_run_id
  FROM nutrichain_lakehouse.bronze.bronze_openfood_products_raw
)
SELECT
  b.ingest_run_id,
  COUNT(*) AS page_rows,
  SUM(COALESCE(json_array_length(b.raw_products_json), 0)) AS product_rows,
  SUM(b.total_products_reported) AS api_reported_products,
  MIN(b.ingested_at) AS run_started_at,
  MAX(b.ingested_at) AS run_finished_at
FROM nutrichain_lakehouse.bronze.bronze_openfood_products_raw b
INNER JOIN latest_run lr ON b.ingest_run_id = lr.ingest_run_id
GROUP BY b.ingest_run_id;

# Bronze: Ingestion Run

In [0]:
%sql
SELECT
    ingest_run_id,
    COUNT(*) AS page_rows,
    SUM(page_size) AS products_in_pages,
    MIN(ingested_at) AS first_ingested_at,
    MAX(ingested_at) AS last_ingested_at
FROM nutrichain_lakehouse.bronze.bronze_openfood_products_raw
GROUP BY ingest_run_id
ORDER BY last_ingested_at DESC;

## Bronze: did each run land?

In [0]:
%sql
SELECT
  ingest_run_id,
  COUNT(*) AS page_rows,
  SUM(page_size) AS products_in_pages,
  MIN(ingested_at) AS first_ingested_at,
  MAX(ingested_at) AS last_ingested_at
FROM nutrichain_lakehouse.bronze.bronze_openfood_products_raw
GROUP BY ingest_run_id
ORDER BY last_ingested_at DESC;

## Explode JSON → see column names

In [0]:
%sql
SELECT
  p.code AS barcode,
  p.product_name,
  p.`energy-kcal_100g`,
  p.energy_100g,
  p.nutriscore_grade,
  p.nova_group,
  p.sugars_100g
FROM nutrichain_lakehouse.bronze.bronze_openfood_products_raw b
LATERAL VIEW explode(
  from_json(
    b.raw_products_json,
    'array<struct<
      code:string,
      product_name:string,
      `energy-kcal_100g`:string,
      energy_100g:string,
      nutriscore_grade:string,
      nova_group:string,
      sugars_100g:string
    >>'
  )
) t AS p
--WHERE b.ingest_run_id = 'YOUR_LATEST_RUN'
LIMIT 25;

## Peek raw JSON (see API fields)

In [0]:
%sql
SELECT
  ingest_run_id,
  api_page_number,
  LEFT(raw_products_json, 400) AS json_start
FROM nutrichain_lakehouse.bronze.bronze_openfood_products_raw
--WHERE ingest_run_id = 'YOUR_LATEST_RUN'
LIMIT 3;